# Phase 4B Lab - Real Search Extraction

Mục tiêu: hiểu real Amazon/BestBuy search được đặt sau opt-in flag như thế nào.

Default expected output: parser/tool tests pass, live tests skipped.

Safety: live search chỉ chạy khi `ENABLE_REAL_SEARCH=true`.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def find_v3_root(start: str | None = None) -> Path:
    path = Path(start or os.getcwd()).resolve()
    for candidate in (path, *path.parents):
        if candidate.name == "shopping_assistant_v3" and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the tech2ai/shopping_assistant_v3 tree.")


V3_ROOT = find_v3_root()
os.chdir(V3_ROOT)
if str(V3_ROOT) not in sys.path:
    sys.path.insert(0, str(V3_ROOT))


def run(command: list[str], timeout: int = 120) -> subprocess.CompletedProcess[str] | None:
    print("$ " + " ".join(command))
    try:
        result = subprocess.run(
            command,
            cwd=V3_ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        print(f"Command timed out after {timeout} seconds.")
        return None

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"exit_code={result.returncode}")
    return result


print(f"V3_ROOT={V3_ROOT}")
print("Default safety flags:")
for name in ("ENABLE_REAL_SEARCH", "ENABLE_REAL_MODEL_CALLS", "ENABLE_AGENTS_SDK"):
    print(f"{name}={os.getenv(name, '<unset>')}")


## 1. Chạy parser + tool tests an toàn

Command này dùng fixtures local, không scrape live.


In [ ]:
run(["uv", "run", "pytest", "tests/test_tools.py", "tests/test_real_search.py", "-q", "--tb=short"], timeout=180)


## 2. Kiểm tra default mock dispatch

Expected: `ENABLE_REAL_SEARCH` mặc định false; `deal_search()` dùng fixture.


In [ ]:
from backend.shared import config
from backend.tools.deal_search.schemas import DealSearchInput
from backend.tools.deal_search.tool import deal_search

print(f"ENABLE_REAL_SEARCH={config.ENABLE_REAL_SEARCH}")
output = deal_search(DealSearchInput(query_en="phone", source="Amazon", max_results_per_source=2))
print(f"products={len(output.products)}")
print(f"warnings={output.warnings}")
for product in output.products[:2]:
    print(product.source, product.title, product.sale_price_usd)


## 3. Opt-in live search cell

Cell này chỉ chạy live test nếu environment đã bật `ENABLE_REAL_SEARCH=true`.
Nếu chưa bật, expected output là skip message.


In [ ]:
if os.getenv("ENABLE_REAL_SEARCH", "").strip().lower() == "true":
    run(["uv", "run", "pytest", "tests/test_real_search.py", "-q", "--tb=short"], timeout=240)
else:
    print("Skipped live search. Set ENABLE_REAL_SEARCH=true only when you want live Amazon/BestBuy requests.")


## 4. Cách đọc kết quả

- Parser tests pass: extraction logic còn parse được fixture HTML.
- Live tests skipped: default behavior đúng.
- Live failures có thể do website/network, không tự coi là regression nếu mock
  tests vẫn pass và warning được sanitize.
